# Data Preprocessing & Synthetic Data Generation for Dashboard

---
## 1. Project Overview

This notebook demonstrates synthetic data generation for a **bank mortgage and consumer financing dashboard** across:
- **KPR Subsidi (S)** — Subsidized home loans
- **KPR Non Subsidi (NS)** — Non-subsidized home loans
- **Non KPR (NK)** — Non-mortgage consumer financing

Metrics include: unit counts, Juta IDR values, RKAP targets, realization, prognosa, and YoY 2024/2025 comparisons across 35 KCS branches in 4 regional clusters.

---
## 2. Library Imports & Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", "{:,.2f}".format)
plt.rcParams["figure.figsize"] = (12, 5)
sns.set_style("whitegrid")

print("Libraries loaded successfully")

Libraries loaded successfully


---
## 3. Original Data Exploration

We explore the original file to understand structure, semantics, and value ranges.

In [ ]:
ORIGINAL_PATH = r"D:\intern-BSN\DUMMY Database.xlsx"

try:
    raw = pd.read_excel(ORIGINAL_PATH, sheet_name=None)
    print(f"Sheets found: {list(raw.keys())}")
    for sheet_name in ["ALL_FIKS", "Historical", "2025", "KCPS"]:
        if sheet_name in raw:
            df = raw[sheet_name]
            print(f"\n{sheet_name}: {df.shape}  |  Columns: {list(df.columns)[:6]}...")
except FileNotFoundError:
    print("Original file not found — proceeding with synthetic generation.")
    raw = {}

---
## 4. Master Reference Data

Define the 35-branch catalog, 4 regional clusters, size tiers, and base metrics.

In [ ]:
BRANCHES = {
    701: ("KCS JKT HARMONI",   "KCS Jakarta Harmoni",      "Jakarta Jabar Banten"),
    702: ("KCS BANDUNG",       "KCS Bandung",              "Jakarta Jabar Banten"),
    703: ("KCS SURABAYA",      "KCS Surabaya",             "Jateng DIY Jatim Nusra"),
    704: ("KCS YOGYAKARTA",    "KCS Yogyakarta",           "Jateng DIY Jatim Nusra"),
    705: ("KCS MAKASSAR",      "KCS Makassar",             "Kalimantan Sulawesi"),
    706: ("KCS MALANG",        "KCS Malang",               "Jateng DIY Jatim Nusra"),
    707: ("KCS SOLO",          "KCS Solo",                 "Jateng DIY Jatim Nusra"),
    708: ("KCS BATAM",         "KCS Batam",                "Sumatera"),
    709: ("KCS MEDAN",         "KCS Medan",                "Sumatera"),
    710: ("KCS TANGERANG",     "KCS Tangerang",            "Jakarta Jabar Banten"),
    711: ("KCS BOGOR",         "KCS Bogor",                "Jakarta Jabar Banten"),
    712: ("KCS BEKASI",        "KCS Bekasi",               "Jakarta Jabar Banten"),
    713: ("KCS PEKANBARU",     "KCS Pekanbaru",            "Sumatera"),
    714: ("KCS SEMARANG",      "KCS Semarang",             "Jateng DIY Jatim Nusra"),
    715: ("KCS BANJARMASIN",   "KCS Banjarmasin",          "Kalimantan Sulawesi"),
    716: ("KCS CIREBON",       "KCS Cirebon",              "Jakarta Jabar Banten"),
    717: ("KCS PALEMBANG",     "KCS Palembang",            "Sumatera"),
    718: ("KCS BALIKPAPAN",    "KCS Balikpapan",           "Kalimantan Sulawesi"),
    719: ("KCS SERANG",        "KCS Serang",               "Jakarta Jabar Banten"),
    720: ("KCS JKT PS MINGGU", "KCS Jakarta Pasar Minggu", "Jakarta Jabar Banten"),
    724: ("KCS DEPOK",         "KCS Depok",                "Jakarta Jabar Banten"),
    731: ("KCS TASIKMALAYA",   "KCS Tasikmalaya",          "Jakarta Jabar Banten"),
    741: ("KCS TEGAL",         "KCS Tegal",                "Jateng DIY Jatim Nusra"),
    752: ("KCS BANDA ACEH",    "KCS Banda Aceh",           "Sumatera"),
    761: ("KCS KARAWANG",      "KCS Karawang",             "Jakarta Jabar Banten"),
    767: ("KCS MATARAM",       "KCS Mataram",              "Jateng DIY Jatim Nusra"),
    773: ("KCS KENDARI",       "KCS Kendari",              "Kalimantan Sulawesi"),
    774: ("KCS PALU",          "KCS Palu",                 "Kalimantan Sulawesi"),
    781: ("KCS BENGKULU",      "KCS Bengkulu",             "Sumatera"),
    801: ("KCS LHOKSEUMAWE",   "KCS Lhokseumawe",          "Sumatera"),
    803: ("KCS JAMBI",         "KCS Jambi",                "Sumatera"),
    804: ("KCS PADANG",        "KCS Padang",               "Sumatera"),
    817: ("KCS LAMPUNG",       "KCS Lampung",              "Sumatera"),
    818: ("KCS PONTIANAK",     "KCS Pontianak",            "Kalimantan Sulawesi"),
    820: ("KCS JEMBER",        "KCS Jember",               "Jateng DIY Jatim Nusra"),
}
KODE_KCS_LIST = sorted(BRANCHES.keys())

# Regional economic performance multipliers
REGIONAL_MULTIPLIER = {
    "Jakarta Jabar Banten":   1.25,
    "Jateng DIY Jatim Nusra": 1.10,
    "Kalimantan Sulawesi":    0.95,
    "Sumatera":               0.85,
}

BRANCH_TIER = {
    701:"large", 702:"large", 703:"large", 704:"medium",
    705:"large", 706:"large", 707:"large", 708:"medium",
    709:"large", 710:"large", 711:"large", 712:"large",
    713:"medium",714:"large", 715:"medium",716:"medium",
    717:"medium",718:"medium",719:"medium",720:"medium",
    724:"medium",731:"small", 741:"small", 752:"small",
    761:"medium",767:"small", 773:"small", 774:"small",
    781:"small", 801:"small", 803:"small", 804:"medium",
    817:"medium",818:"small", 820:"medium",
}

TIER_BASE = {
    "large":  {"S_units":2000,"NS_units":700, "NK_units":60, "S_avg_jt":160,"NS_avg_jt":350,"NK_avg_jt":200},
    "medium": {"S_units":1000,"NS_units":350, "NK_units":35, "S_avg_jt":155,"NS_avg_jt":330,"NK_avg_jt":190},
    "small":  {"S_units":400, "NS_units":150, "NK_units":20, "S_avg_jt":150,"NS_avg_jt":300,"NK_avg_jt":180},
}

def noise(scale=0.08):
    """Gaussian multiplicative noise: adds realistic variability around 1."""
    return 1 + np.random.normal(0, scale)

branch_df = pd.DataFrame(
    [(k, v[0], v[2], BRANCH_TIER[k]) for k, v in BRANCHES.items()],
    columns=["KODE_KCS", "CABANG", "WILAYAH", "TIER"]
)
print(f"Total KCS branches: {len(BRANCHES)}")
branch_df.groupby(["WILAYAH","TIER"]).size().unstack(fill_value=0)

---
## 5. Sheet: ALL_FIKS – KCS Branch Summary

In [ ]:
def gen_all_fiks():
    """
    KCS-level summary with:
    - NOV 2025 + December incremental realization
    - DES 2025 cumulative (NOV + DEC)
    - RKAP 2025 targets (±15% variance from realization)
    - Achievement ratios (REAL/RKAP, REAL/PROG)
    - Prognosa forecast values
    - YoY 2024 actuals
    """
    rows = []
    for kode in KODE_KCS_LIST:
        short_name, _, wilayah = BRANCHES[kode]
        tier = BRANCH_TIER[kode]; base = TIER_BASE[tier]
        mult = REGIONAL_MULTIPLIER[wilayah]

        S_avg  = base["S_avg_jt"]  * noise(0.05)
        NS_avg = base["NS_avg_jt"] * noise(0.05)
        NK_avg = base["NK_avg_jt"] * noise(0.05)

        S_nov_u  = max(10, int(base["S_units"]  * mult * noise()))
        NS_nov_u = max(5,  int(base["NS_units"] * mult * noise()))
        NK_nov_u = max(2,  int(base["NK_units"] * mult * noise()))

        S_nov_jt  = round(S_nov_u  * S_avg,  2)
        NS_nov_jt = round(NS_nov_u * NS_avg, 2)
        NK_nov_jt = round(NK_nov_u * NK_avg, 2)

        S_curr_u  = max(5, int(S_nov_u  * 0.08 * noise(0.2)))
        NS_curr_u = max(2, int(NS_nov_u * 0.06 * noise(0.2)))
        NK_curr_u = max(1, int(NK_nov_u * 0.07 * noise(0.2)))

        S_curr_jt  = round(S_curr_u  * S_avg  * noise(0.05), 2)
        NS_curr_jt = round(NS_curr_u * NS_avg * noise(0.05), 2)
        NK_curr_jt = round(NK_curr_u * NK_avg * noise(0.05), 2)

        S_des25_u  = S_nov_u + S_curr_u
        NS_des25_u = NS_nov_u + NS_curr_u
        NK_des25_u = NK_nov_u + NK_curr_u

        S_des25_jt  = round(S_nov_jt  + S_curr_jt,  2)
        NS_des25_jt = round(NS_nov_jt + NS_curr_jt, 2)
        NK_des25_jt = round(NK_nov_jt + NK_curr_jt, 2)

        S_rkap  = round(S_des25_jt  * np.random.uniform(0.88, 1.15), 2)
        NS_rkap = round(NS_des25_jt * np.random.uniform(0.85, 1.20), 2)
        NK_rkap = round(NK_des25_jt * np.random.uniform(0.80, 1.25), 2)

        S_prog  = max(5000,  int(S_rkap  * np.random.uniform(0.90, 1.05)))
        NS_prog = max(5000,  int(NS_rkap * np.random.uniform(0.88, 1.08)))
        NK_prog = max(500,   int(NK_rkap * np.random.uniform(0.85, 1.10)))

        S_des24_u   = max(5, int(S_des25_u  * np.random.uniform(0.60, 0.82)))
        NS_des24_u  = max(2, int(NS_des25_u * np.random.uniform(0.80, 1.05)))
        NK_des24_u  = max(1, int(NK_des25_u * np.random.uniform(0.50, 0.75)))
        S_des24_jt  = round(S_des24_u  * S_avg  * np.random.uniform(0.95, 1.05), 2)
        NS_des24_jt = round(NS_des24_u * NS_avg * np.random.uniform(0.93, 1.05), 2)
        NK_des24_jt = round(NK_des24_u * NK_avg * np.random.uniform(0.90, 1.05), 2)

        rows.append({
            "KODE_KCS": kode, "WILAYAH": wilayah, "CABANG": short_name,
            "S_NOV_25_(U)": S_nov_u, "S_NOV_25_(JT)": S_nov_jt,
            "current_S_25_(U)": S_curr_u, "current_S_25_(JT)": S_curr_jt,
            "S_DES_25_(U)": S_des25_u, "S_DES_25_(JT)": S_des25_jt,
            "S_RKAP_25_(JT)": S_rkap, "S_REAL_DES_25_RKAP_(%)": round(S_des25_jt/S_rkap, 6),
            "S_PROG_(JT)": S_prog, "S_REAL_PROG_(%)": round(S_des25_jt/S_prog, 6),
            "S_REAL_SUM_(%)": round(S_des25_jt/9_800_000, 6),
            "NS_NOV_25_(U)": NS_nov_u, "NS_NOV_25_(JT)": NS_nov_jt,
            "current_NS_25_(U)": NS_curr_u, "current_NS_25_(JT)": NS_curr_jt,
            "NS_DES_25_(U)": NS_des25_u, "NS_DES_25_(JT)": NS_des25_jt,
            "NS_RKAP_25_(JT)": NS_rkap, "NS_REAL_DES_25_RKAP_(%)": round(NS_des25_jt/NS_rkap, 6),
            "NS_PROG_(JT)": NS_prog, "NS_REAL_PROG_(%)": round(NS_des25_jt/NS_prog, 6),
            "NS_REAL_SUM_(%)": round(NS_des25_jt/4_100_000, 6),
            "NK_NOV_25_(U)": NK_nov_u, "NK_NOV_25_(JT)": NK_nov_jt,
            "current_NK_25_(U)": NK_curr_u, "current_NK_25_(JT)": NK_curr_jt,
            "NK_DES_25_(U)": NK_des25_u, "NK_DES_25_(JT)": NK_des25_jt,
            "NK_RKAP_25_(JT)": NK_rkap, "NK_REAL_DES_25_RKAP_(%)": round(NK_des25_jt/NK_rkap, 6),
            "NK_PROG_(JT)": NK_prog, "NK_REAL_PROG_(%)": round(NK_des25_jt/NK_prog, 6),
            "NK_REAL_SUM_(%)": round(NK_des25_jt/315_000, 6),
            "S_DES_24_(U)": S_des24_u, "S_DES_24_(JT)": S_des24_jt,
            "NS_DES_24_(U)": NS_des24_u, "NS_DES_24_(JT)": NS_des24_jt,
            "NK_DES_24_(U)": NK_des24_u, "NK_DES_24_(JT)": NK_des24_jt,
        })
    return pd.DataFrame(rows)

df_allfiks = gen_all_fiks()
print(f"ALL_FIKS shape: {df_allfiks.shape}")
df_allfiks[["KODE_KCS","WILAYAH","CABANG","S_DES_25_(JT)","S_REAL_DES_25_RKAP_(%)"]].head(8)

In [ ]:
# RKAP Achievement Distribution
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, title, color in zip(
    axes,
    ["S_REAL_DES_25_RKAP_(%)", "NS_REAL_DES_25_RKAP_(%)", "NK_REAL_DES_25_RKAP_(%)"],
    ["KPR Subsidi", "KPR Non Subsidi", "Non KPR"],
    ["#2196F3", "#4CAF50", "#FF9800"]
):
    top = df_allfiks.nlargest(10, col)
    ax.barh(top["CABANG"], top[col], color=color, alpha=0.85)
    ax.axvline(1.0, color="red", linestyle="--", lw=1.5, label="100% Target")
    ax.set_title(f"{title} Achievement", fontsize=10, fontweight="bold")
    ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
    ax.legend(fontsize=8)

plt.suptitle("Top 10 Branches RKAP Achievement — DES 2025", fontsize=12, fontweight="bold")
plt.tight_layout(); plt.show()

---
## 6. Sheet: Historical – Monthly Time-Series

In [ ]:
def gen_historical():
    """
    1,260 rows: 35 branches × 12 months × 3 products
    - Seasonal multipliers (Jan low → Dec peak)
    - 1.2%/month upward trend
    - 12% Gaussian noise for natural variability
    """
    months = pd.date_range("2025-01-01", "2025-12-01", freq="MS")
    products = ["KPR Subsidi", "KPR Non Subsidi", "Non KPR"]
    seasonal = {
        1:0.72, 2:0.80, 3:0.88, 4:0.92, 5:0.96, 6:1.05,
        7:1.08, 8:1.05, 9:1.00, 10:1.02, 11:1.10, 12:1.20,
    }

    rows = []
    for kode in KODE_KCS_LIST:
        short_name, _, wilayah = BRANCHES[kode]
        tier = BRANCH_TIER[kode]; base = TIER_BASE[tier]
        mult = REGIONAL_MULTIPLIER[wilayah]
        annual = {
            "KPR Subsidi":     base["S_units"]  * base["S_avg_jt"]  * mult,
            "KPR Non Subsidi": base["NS_units"] * base["NS_avg_jt"] * mult,
            "Non KPR":         base["NK_units"] * base["NK_avg_jt"] * mult,
        }
        for prod in products:
            monthly_base = annual[prod] / 12
            for month in months:
                m = month.month
                trend = 1 + (m - 1) * 0.012
                val = max(0, round(monthly_base * seasonal[m] * trend * noise(0.12)))
                rows.append({"BULAN":month,"KODE_KCS":kode,"PRODUK":prod,
                              "REALISASI_JT":int(val),"CABANG":short_name})

    return pd.DataFrame(rows).sort_values(["BULAN","KODE_KCS","PRODUK"]).reset_index(drop=True)

df_historical = gen_historical()
print(f"Historical shape: {df_historical.shape}")
df_historical.head(6)

In [ ]:
# Monthly trend chart
monthly_nat = df_historical.groupby(["BULAN","PRODUK"])["REALISASI_JT"].sum().reset_index()
monthly_nat["REALISASI_T"] = monthly_nat["REALISASI_JT"] / 1e6

fig, ax = plt.subplots(figsize=(13, 5))
colors = {"KPR Subsidi":"#1565C0", "KPR Non Subsidi":"#2E7D32", "Non KPR":"#E65100"}
for prod, grp in monthly_nat.groupby("PRODUK"):
    ax.plot(grp["BULAN"], grp["REALISASI_T"], marker="o", label=prod,
            color=colors[prod], linewidth=2)
    ax.fill_between(grp["BULAN"], grp["REALISASI_T"], alpha=0.08, color=colors[prod])
ax.set_title("National Monthly Realization 2025 (Triliun IDR)", fontsize=13, fontweight="bold")
ax.set_ylabel("Triliun IDR"); ax.legend()
plt.xticks(rotation=30); plt.tight_layout(); plt.show()

---
## 7. Sheet: 2025 – National KPI Snapshot

In [ ]:
def gen_2025(df_allfiks):
    """Single-row national summary: realization + portfolio position for 2024 & 2025."""
    real_S_24  = int(df_allfiks["S_DES_24_(JT)"].sum())
    real_NS_24 = int(df_allfiks["NS_DES_24_(JT)"].sum())
    real_NK_24 = int(df_allfiks["NK_DES_24_(JT)"].sum())
    real_S_25  = round(df_allfiks["S_DES_25_(JT)"].sum(), 2)
    real_NS_25 = round(df_allfiks["NS_DES_25_(JT)"].sum(), 2)
    real_NK_25 = round(df_allfiks["NK_DES_25_(JT)"].sum(), 2)
    # Outstanding portfolio position >> annual production
    return pd.DataFrame([{
        "REAL_S_DES_24_(JT)": real_S_24, "REAL_NS_DES_24_(JT)": real_NS_24,
        "REAL_NK_DES_24_(JT)": real_NK_24,
        "POS_S_NOV_24_(JT)":  int(real_S_24  * 4.65 * noise(0.03)),
        "POS_NS_NOV_24_(JT)": int(real_NS_24 * 3.45 * noise(0.03)),
        "POS_NK_NOV_24_(JT)": int(real_NK_24 * 1.35 * noise(0.03)),
        "REAL_S_DES_25_(JT)": real_S_25, "REAL_NS_DES_25_(JT)": real_NS_25,
        "REAL_NK_DES_25_(JT)": real_NK_25,
        "POS_S_NOV_25_(JT)":  int(real_S_25  * 3.48 * noise(0.03)),
        "POS_NS_NOV_25_(JT)": int(real_NS_25 * 4.20 * noise(0.03)),
        "POS_NK_NOV_25_(JT)": int(real_NK_25 * 1.70 * noise(0.03)),
    }])

df_2025 = gen_2025(df_allfiks)
print("2025 National KPI:")
df_2025.T.rename(columns={0: "Value (Juta IDR)"})

---
## 8. Sheet: KCPS – Sub-Branch Detail

In [ ]:
def gen_kcps(df_allfiks):
    """KCPS mirrors ALL_FIKS structure with exact values for dashboard drill-down."""
    rows = []
    for _, row in df_allfiks.iterrows():
        kode = row["KODE_KCS"]; long_name = BRANCHES[kode][1]
        S_jt  = row["S_DES_25_(JT)"]; NS_jt = row["NS_DES_25_(JT)"]; NK_jt = row["NK_DES_25_(JT)"]
        S_u   = row["S_DES_25_(U)"];  NS_u  = row["NS_DES_25_(U)"];  NK_u  = row["NK_DES_25_(U)"]
        S_rkap  = round(S_jt  * np.random.uniform(0.88, 1.15), 2)
        NS_rkap = round(NS_jt * np.random.uniform(0.85, 1.20), 2)
        NK_rkap = round(NK_jt * np.random.uniform(0.80, 1.25), 2)
        S_prog  = max(1000, int(S_rkap  * np.random.uniform(0.88, 1.08)))
        NS_prog = max(500,  int(NS_rkap * np.random.uniform(0.85, 1.10)))
        NK_prog = max(100,  int(NK_rkap * np.random.uniform(0.80, 1.15)))
        rows.append({
            "KODE_KCS":kode,"KODE_CABANG":kode,"CABANG":long_name,
            "S_NOV_25_(U)":int(S_u*0.92), "S_NOV_25_(JT)":round(S_jt*0.92,2),
            "current_S_25_(U)":int(S_u*0.08), "current_S_25_(JT)":round(S_jt*0.08,2),
            "S_DES_25_(U)":S_u, "S_DES_25_(JT)":S_jt,
            "S_RKAP_25_(JT)":S_rkap, "S_REAL_DES_25_RKAP_(%)": round(S_jt/S_rkap,6),
            "S_PROG_(JT)":S_prog, "S_REAL_PROG_(%)": round(S_jt/S_prog,6),
            "S_REAL_SUM_(%)": round(S_jt/9_800_000,6),
            "NS_NOV_25_(U)":int(NS_u*0.92), "NS_NOV_25_(JT)":round(NS_jt*0.92,2),
            "current_NS_25_(U)":int(NS_u*0.08), "current_NS_25_(JT)":round(NS_jt*0.08,2),
            "NS_DES_25_(U)":NS_u, "NS_DES_25_(JT)":NS_jt,
            "NS_RKAP_25_(JT)":NS_rkap, "NS_REAL_DES_25_RKAP_(%)": round(NS_jt/NS_rkap,6),
            "NS_PROG_(JT)":NS_prog, "NS_REAL_PROG_(%)": round(NS_jt/NS_prog,6),
            "NS_REAL_SUM_(%)": round(NS_jt/4_100_000,6),
            "NK_NOV_25_(U)":int(NK_u*0.92), "NK_NOV_25_(JT)":round(NK_jt*0.92,2),
            "current_NK_25_(U)":int(NK_u*0.08), "current_NK_25_(JT)":round(NK_jt*0.08,2),
            "NK_DES_25_(U)":NK_u, "NK_DES_25_(JT)":NK_jt,
            "NK_RKAP_25_(JT)":NK_rkap, "NK_REAL_DES_25_RKAP_(%)": round(NK_jt/NK_rkap,6),
            "NK_PROG_(JT)":NK_prog, "NK_REAL_PROG_(%)": round(NK_jt/NK_prog,6),
            "NK_REAL_SUM_(%)": round(NK_jt/315_000,6),
        })
    return pd.DataFrame(rows)

df_kcps = gen_kcps(df_allfiks)
print(f"KCPS shape: {df_kcps.shape}")
df_kcps[["KODE_KCS","CABANG","S_DES_25_(JT)","S_REAL_DES_25_RKAP_(%)"]].head(5)

---
## 9. Export to Excel with Professional Formatting

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

HEADER_FILL = PatternFill("solid", start_color="003366", end_color="003366")
ALT_FILL    = PatternFill("solid", start_color="EBF3FB", end_color="EBF3FB")
HEADER_FONT = Font(bold=True, color="FFFFFF", name="Arial", size=9)
DATA_FONT   = Font(name="Arial", size=9)
CENTER = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT   = Alignment(horizontal="left",  vertical="center")
RIGHT  = Alignment(horizontal="right", vertical="center")
THIN   = Side(style="thin", color="BFBFBF")
BORDER = Border(left=THIN, right=THIN, top=THIN, bottom=THIN)

def write_df_to_sheet(ws, df, pct_cols=None, int_cols=None, jt_cols=None):
    pct_cols = pct_cols or []; int_cols = int_cols or []; jt_cols = jt_cols or []
    for c_idx, col in enumerate(df.columns, 1):
        cell = ws.cell(row=1, column=c_idx, value=col)
        cell.fill=HEADER_FILL; cell.font=HEADER_FONT; cell.alignment=CENTER; cell.border=BORDER
    for r_idx, row_data in enumerate(df.itertuples(index=False), 2):
        fill = ALT_FILL if r_idx % 2 == 0 else PatternFill()
        for c_idx, (col, val) in enumerate(zip(df.columns, row_data), 1):
            cell = ws.cell(row=r_idx, column=c_idx, value=val)
            cell.font=DATA_FONT; cell.border=BORDER; cell.fill=fill
            if   col in pct_cols: cell.number_format="0.00%"; cell.alignment=RIGHT
            elif col in int_cols: cell.number_format="#,##0";  cell.alignment=RIGHT
            elif col in jt_cols:  cell.number_format="#,##0";  cell.alignment=RIGHT
            elif isinstance(val, float): cell.number_format="#,##0.00"; cell.alignment=RIGHT
            elif isinstance(val, int):   cell.number_format="#,##0";    cell.alignment=RIGHT
            else: cell.alignment=LEFT
    for c_idx, col in enumerate(df.columns, 1):
        ws.column_dimensions[get_column_letter(c_idx)].width = min(len(str(col))+3, 22)
    ws.freeze_panes="A2"; ws.row_dimensions[1].height=30

OUTPUT_PATH = "dummy_dashboard_data.xlsx"
wb = Workbook(); wb.remove(wb.active)

ws1 = wb.create_sheet("ALL_FIKS")
write_df_to_sheet(ws1, df_allfiks,
    pct_cols=[c for c in df_allfiks.columns if "REAL" in c and "%" in c],
    int_cols=[c for c in df_allfiks.columns if "_(U)" in c],
    jt_cols =[c for c in df_allfiks.columns if "_(JT)" in c and "REAL" not in c])

ws2 = wb.create_sheet("Historical")
write_df_to_sheet(ws2, df_historical, int_cols=["KODE_KCS","REALISASI_JT"])

ws3 = wb.create_sheet("2025")
write_df_to_sheet(ws3, df_2025, int_cols=list(df_2025.columns))

ws4 = wb.create_sheet("KCPS")
write_df_to_sheet(ws4, df_kcps,
    pct_cols=[c for c in df_kcps.columns if "REAL" in c and "%" in c],
    int_cols=[c for c in df_kcps.columns if "_(U)" in c],
    jt_cols =[c for c in df_kcps.columns if "_(JT)" in c and "REAL" not in c])

wb.save(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH}  |  Sheets: {wb.sheetnames}")

---
## 10. Validation & Quality Checks

In [ ]:
print("=" * 60)
print("DATA QUALITY VALIDATION REPORT")
print("=" * 60)
passed = 0; total = 0

def check(name, cond, detail=""):
    global passed, total; total += 1
    status = "PASS" if cond else "FAIL"
    if cond: passed += 1
    print(f"[{status}]  {name}")
    if detail: print(f"       {detail}")

check("ALL_FIKS: 35 rows", len(df_allfiks) == 35)
check("ALL_FIKS: No nulls", df_allfiks.isnull().sum().sum() == 0)
check("ALL_FIKS: Units positive", (df_allfiks[[c for c in df_allfiks.columns if "_(U)" in c]] > 0).all().all())
check("ALL_FIKS: DES_25 = NOV_25 + current",
    (df_allfiks["S_DES_25_(U)"] == df_allfiks["S_NOV_25_(U)"] + df_allfiks["current_S_25_(U)"]).all())
check("ALL_FIKS: RKAP achievement 0.5–1.8",
    ((df_allfiks["S_REAL_DES_25_RKAP_(%)"] > 0.5) & (df_allfiks["S_REAL_DES_25_RKAP_(%)"] < 1.8)).all())
check("ALL_FIKS: 4 regions", df_allfiks["WILAYAH"].nunique() == 4)
check("Historical: 1,260 rows", len(df_historical) == 1260)
check("Historical: No nulls", df_historical.isnull().sum().sum() == 0)
check("Historical: 3 products", df_historical["PRODUK"].nunique() == 3)
check("Historical: 35 branches", df_historical["CABANG"].nunique() == 35)
check("2025: 1 row", len(df_2025) == 1)
check("2025: POS > REAL (portfolio > production)",
    df_2025["POS_S_NOV_25_(JT)"].iloc[0] > df_2025["REAL_S_DES_25_(JT)"].iloc[0])
check("KCPS: 35 rows", len(df_kcps) == 35)
check("KCPS: S values match ALL_FIKS",
    (df_kcps["S_DES_25_(JT)"].values == df_allfiks["S_DES_25_(JT)"].values).all())

print(f"\n{'='*60}")
print(f"RESULT: {passed}/{total} checks passed")
if passed == total:
    print("All validation checks PASSED — dataset is dashboard-ready!")

In [ ]:
# Final summary statistics
print("\nDATASET SUMMARY")
print("-" * 50)
total_S  = df_allfiks["S_DES_25_(JT)"].sum()
total_NS = df_allfiks["NS_DES_25_(JT)"].sum()
total_NK = df_allfiks["NK_DES_25_(JT)"].sum()
print(f"National DES 2025 Realization:")
print(f"  KPR Subsidi    : Rp {total_S/1e6:>7,.1f} Triliun")
print(f"  KPR Non Subsidi: Rp {total_NS/1e6:>7,.1f} Triliun")
print(f"  Non KPR        : Rp {total_NK/1e6:>7,.1f} Triliun")
print(f"  TOTAL          : Rp {(total_S+total_NS+total_NK)/1e6:>7,.1f} Triliun")
yoy_s = total_S/df_allfiks["S_DES_24_(JT)"].sum()-1
yoy_ns= total_NS/df_allfiks["NS_DES_24_(JT)"].sum()-1
yoy_nk= total_NK/df_allfiks["NK_DES_24_(JT)"].sum()-1
print(f"\nYoY Growth 2024 → 2025: S={yoy_s:+.1%} | NS={yoy_ns:+.1%} | NK={yoy_nk:+.1%}")
print(f"\nFile saved: dummy_dashboard_data.xlsx")

---

## Dashboard Integration Guide (Looker Studio)

| Sheet | Primary Use Case |
|-------|------------------|
| **ALL_FIKS** | KCS branch KPIs, achievement ratios, YoY scorecards |
| **Historical** | Monthly trend lines by product and region |
| **2025** | National KPI header cards (totals + portfolio position) |
| **KCPS** | Sub-branch drill-down analysis |

**Recommended join key:** `KODE_KCS` across all sheets

**Calculated fields for dashboards:**
- Achievement: `S_DES_25_(JT) / S_RKAP_25_(JT)`
- YoY Growth: `(S_DES_25_(JT) - S_DES_24_(JT)) / S_DES_24_(JT)`
- Prognosa Gap: `S_DES_25_(JT) - S_PROG_(JT)`
